In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# Pharma Sales Analytics
# ========================================

from pyspark.sql.functions import *

# Read Pharmacy Data
pharma_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(f"{source_path}/pharmacy")
)

# Medicine Sales Analytics
pharma_sales_df = pharma_df.groupBy(
    "medicine_id"
).agg(
    round(sum("total_amount"), 2).alias("total_sales"),
    count("transaction_id").alias("total_transactions")
)

# Simple Forecast (10% Growth)
pharma_sales_df = pharma_sales_df.withColumn(
    "forecast_sales",
    round(col("total_sales") * 1.10, 2)
)

# Write Gold Layer
pharma_sales_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/pharma_sales_forecasting")

display(pharma_sales_df)
print("Pharma Sales Analytics Completed Successfully")